# Crop Disease Detection — EfficientNet-B0 Training Pipeline
Trains an EfficientNet-B0 model with:
- 2-phase transfer learning (3 frozen epochs -> cosine fine-tuning)
- SHA-256 deduplication and deterministic stratified 10% test split (matching backend evaluation script)
- Mixed precision (`torch.amp.autocast`) & phone photo augmentations
- Class-weighted loss for imbalanced classes
- Temperature scaling calibration on validation set
- Full evaluation export: confusion matrix PNG, JSON report, and `.pth` checkpoint with `trained: True`

In [ ]:
# ── Cell 1: GPU Availability Check ───────────────────────────────────────────
import torch
import sys

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    error_msg = (
        "\n" + "=" * 70 + "\n"
        "[ERROR] No GPU detected!\n"
        "Please enable GPU accelerator in Kaggle:\n"
        "Notebook settings -> Accelerator -> GPU T4 x2 (or P100).\n"
        + "=" * 70
    )
    print(error_msg, file=sys.stderr)
    raise RuntimeError("Training halted: GPU is required but not available.")

gpu_count = torch.cuda.device_count()
for i in range(gpu_count):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU [{i}]: {p.name} | VRAM: {p.total_memory / (1024**3):.2f} GB")

In [ ]:
# ── Cell 2: Dependencies ──────────────────────────────────────────────────────
!pip install -q --upgrade albumentations scikit-learn matplotlib seaborn tqdm opencv-python-headless
print("Dependencies configured successfully.")

In [ ]:
# ── Cell 3: Imports & Hyperparameter Configuration ───────────────────────────
import os
import sys
import json
import hashlib
import random
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import cv2
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as tv_models
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning)

# Seed for full reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Output directory
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 64
HEAD_LR = 1e-3
FINETUNE_LR = 3e-4
WEIGHT_DECAY = 1e-4
PHASE_A_EPOCHS = 3
PHASE_B_EPOCHS = 20
PATIENCE = 5
TEST_SIZE = 0.10
VAL_SIZE = 0.10
NUM_WORKERS = 4

DEVICE = torch.device("cuda")
print("Execution environment initialized on CUDA.")

In [ ]:
# ── Cell 4: Auto-detect Kaggle Input Dataset Path ──────────────────────────────
KAGGLE_INPUT = Path("/kaggle/input")

def locate_dataset_dir(base_path: Path) -> Path:
    """Search for directory containing the 22 crop disease/pest class folders."""
    image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
    
    def has_image_files(d: Path) -> bool:
        return any(f.suffix.lower() in image_extensions for f in d.iterdir() if f.is_file())
    
    def count_valid_class_dirs(d: Path) -> int:
        return sum(1 for c in d.iterdir() if c.is_dir() and has_image_files(c))
    
    # Check base directory and subdirectories up to 3 levels deep
    for p in base_path.glob("**/*"):
        if p.is_dir() and count_valid_class_dirs(p) >= 15:
            return p
            
    raise FileNotFoundError(
        f"Could not auto-detect crop disease dataset under {base_path}. "
        "Please ensure 'nirmalsankalana/crop-pest-and-disease-detection' is attached."
    )

DATASET_ROOT = locate_dataset_dir(KAGGLE_INPUT)
class_folders = sorted([d.name for d in DATASET_ROOT.iterdir() if d.is_dir() and not d.name.startswith(".")])
print(f"Located dataset root: {DATASET_ROOT}")
print(f"Found {len(class_folders)} class directories:")
for idx, name in enumerate(class_folders):
    print(f"  [{idx:02d}] {name}")

In [ ]:
# ── Cell 5: Deterministic Sorted File Collection, SHA-256 Dedup & Split ──────
# Must precisely match backend/scripts/evaluate_classifier.py:
# 1. Sorted class names
# 2. Sorted file paths per class
# 3. Stratified test split (test_size=0.10, seed=42)

image_extensions = {".jpg", ".jpeg", ".png", ".webp"}
class_dirs = sorted([d.name for d in DATASET_ROOT.iterdir() if d.is_dir() and not d.name.startswith(".")])
class_to_idx = {name: i for i, name in enumerate(class_dirs)}
idx_to_class = {i: name for i, name in enumerate(class_dirs)}

raw_paths: List[str] = []
raw_labels: List[int] = []

for cls_name in class_dirs:
    cls_dir = DATASET_ROOT / cls_name
    cls_idx = class_to_idx[cls_name]
    sorted_files = sorted([
        str(p) for p in cls_dir.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ])
    raw_paths.extend(sorted_files)
    raw_labels.extend([cls_idx] * len(sorted_files))

print(f"Total raw image files discovered: {len(raw_paths)}")

# SHA-256 Deduplication
def calculate_sha256(filepath: str) -> str:
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

print("Computing SHA-256 checksums to filter duplicate images...")
seen_hashes: Dict[str, str] = {}
dedup_paths: List[str] = []
dedup_labels: List[int] = []
dup_count = 0

for path, label in tqdm(zip(raw_paths, raw_labels), total=len(raw_paths), desc="Hashing"):
    try:
        fhash = calculate_sha256(path)
    except OSError:
        continue
    if fhash in seen_hashes:
        dup_count += 1
    else:
        seen_hashes[fhash] = path
        dedup_paths.append(path)
        dedup_labels.append(label)

print(f"Duplicates removed: {dup_count} | Unique images retained: {len(dedup_paths)}")

# Stratified 90% Train+Val / 10% Test split (seed=42)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    dedup_paths, dedup_labels, test_size=TEST_SIZE, random_state=SEED, stratify=dedup_labels
)

# Stratified Train / Val split from remainder
adjusted_val_size = VAL_SIZE / (1.0 - TEST_SIZE)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=adjusted_val_size, random_state=SEED, stratify=train_val_labels
)

# Validate zero split overlap
assert len(set(train_paths) & set(val_paths)) == 0, "Data leak: train/val overlap!"
assert len(set(train_paths) & set(test_paths)) == 0, "Data leak: train/test overlap!"
assert len(set(val_paths) & set(test_paths)) == 0, "Data leak: val/test overlap!"

print(f"Splits established: Train={len(train_paths)} | Val={len(val_paths)} | Held-out Test={len(test_paths)}")

# Compute class weights for imbalanced classes
n_classes = len(class_dirs)
n_train_samples = len(train_labels)
counts_per_class = [train_labels.count(i) for i in range(n_classes)]
class_weights = [n_train_samples / (n_classes * max(1, c)) for c in counts_per_class]

print("\nClass counts and loss weights (Train Split):")
for i, cname in enumerate(class_dirs):
    print(f"  [{i:02d}] {cname:<32} Count={counts_per_class[i]:4d} | Weight={class_weights[i]:.3f}")

In [ ]:
# ── Cell 6: Augmentation Pipeline & PyTorch Dataset ──────────────────────────
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

# Robust phone-photo augmentations
train_transform = A.Compose([
    A.RandomResizedCrop(height=IMG_SIZE, width=IMG_SIZE, scale=(0.5, 1.0), ratio=(0.75, 1.33), p=1.0),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, border_mode=cv2.BORDER_REFLECT, p=0.5),
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1, p=0.7),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 7), p=1.0),
        A.MotionBlur(blur_limit=7, p=1.0),
        A.MedianBlur(blur_limit=5, p=1.0),
    ], p=0.4),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, min_holes=1, min_height=8, min_width=8, fill_value=0, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

class LeafDataset(Dataset):
    def __init__(self, paths: List[str], labels: List[int], transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self) -> int:
        return len(self.paths)

    def _load_image(self, p: str) -> Optional[np.ndarray]:
        img = cv2.imread(p)
        if img is not None:
            return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        try:
            with Image.open(p) as pil_img:
                return np.array(pil_img.convert("RGB"))
        except Exception:
            return None

    def __getitem__(self, idx: int):
        img = self._load_image(self.paths[idx])
        if img is None:
            return self.__getitem__((idx + 1) % len(self))
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, self.labels[idx]

train_ds = LeafDataset(train_paths, train_labels, transform=train_transform)
val_ds = LeafDataset(val_paths, val_labels, transform=val_transform)
test_ds = LeafDataset(test_paths, test_labels, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"DataLoaders created (batch_size={BATCH_SIZE}, num_workers={NUM_WORKERS})")

In [ ]:
# ── Cell 7: Model Definition & Training Utilities ────────────────────────────
def create_efficientnet_b0(num_classes: int) -> nn.Module:
    weights = tv_models.EfficientNet_B0_Weights.DEFAULT
    model = tv_models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def set_backbone_freeze(model: nn.Module, freeze: bool = True):
    for name, param in model.named_parameters():
        if not name.startswith("classifier"):
            param.requires_grad = not freeze

class EarlyStoppingHandler:
    def __init__(self, patience: int = 5, min_delta: float = 1e-4, save_path: Path = OUTPUT_DIR / "_best_weights.pth"):
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.counter = 0
        self.best_loss = float("inf")
        self.should_stop = False

    def step(self, val_loss: float, model: nn.Module) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), self.save_path)
            return True
        self.counter += 1
        if self.counter >= self.patience:
            self.should_stop = True
        return False

def train_or_eval_epoch(model, dataloader, criterion, optimizer, scaler, device, is_training: bool):
    model.train(is_training)
    total_loss, correct, total = 0.0, 0, 0
    desc = "Train" if is_training else "Val"
    pbar = tqdm(dataloader, desc=desc, leave=False)
    
    for images, labels in pbar:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with autocast(device_type="cuda", enabled=True):
            logits = model(images)
            loss = criterion(logits, labels)

        if is_training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        batch_sz = images.size(0)
        total_loss += loss.item() * batch_sz
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += batch_sz
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    return total_loss / total, correct / total

model = create_efficientnet_b0(n_classes).to(DEVICE)
loss_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.05)
print(f"Model initialized on {DEVICE} with {n_classes} target classes.")

In [ ]:
# ── Cell 8: Phase A Training (3 Epochs, Classifier Head Only) ─────────────────
set_backbone_freeze(model, freeze=True)
trainable_params_a = [p for p in model.parameters() if p.requires_grad]
print(f"Phase A: Backbone frozen. Trainable parameters: {sum(p.numel() for p in trainable_params_a):,}")

optimizer_a = optim.AdamW(trainable_params_a, lr=HEAD_LR, weight_decay=WEIGHT_DECAY)
scaler = GradScaler()
phase_a_history = []

print("\n--- Starting Phase A Training ---")
for epoch in range(1, PHASE_A_EPOCHS + 1):
    tr_loss, tr_acc = train_or_eval_epoch(model, train_loader, criterion, optimizer_a, scaler, DEVICE, is_training=True)
    with torch.no_grad():
        va_loss, va_acc = train_or_eval_epoch(model, val_loader, criterion, optimizer_a, scaler, DEVICE, is_training=False)
    
    record = {"phase": "A", "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc}
    phase_a_history.append(record)
    print(f"[Phase A - Epoch {epoch}/{PHASE_A_EPOCHS}] Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc*100:.2f}% | Val Loss: {va_loss:.4f}, Val Acc: {va_acc*100:.2f}%")

print("Phase A complete.")

In [ ]:
# ── Cell 9: Phase B Training (Full Fine-Tuning + Cosine LR + Early Stopping) ─
set_backbone_freeze(model, freeze=False)
print(f"Phase B: Unfrozen all layers. Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

optimizer_b = optim.AdamW(model.parameters(), lr=FINETUNE_LR, weight_decay=WEIGHT_DECAY)
scheduler_b = optim.lr_scheduler.CosineAnnealingLR(optimizer_b, T_max=PHASE_B_EPOCHS, eta_min=1e-6)
early_stopper = EarlyStoppingHandler(patience=PATIENCE, save_path=OUTPUT_DIR / "_best_weights.pth")
phase_b_history = []

print("\n--- Starting Phase B Fine-Tuning ---")
for epoch in range(1, PHASE_B_EPOCHS + 1):
    tr_loss, tr_acc = train_or_eval_epoch(model, train_loader, criterion, optimizer_b, scaler, DEVICE, is_training=True)
    with torch.no_grad():
        va_loss, va_acc = train_or_eval_epoch(model, val_loader, criterion, optimizer_b, scaler, DEVICE, is_training=False)

    improved = early_stopper.step(va_loss, model)
    scheduler_b.step()
    current_lr = optimizer_b.param_groups[0]["lr"]

    record = {"phase": "B", "epoch": epoch, "train_loss": tr_loss, "train_acc": tr_acc, "val_loss": va_loss, "val_acc": va_acc, "lr": current_lr, "improved": improved}
    phase_b_history.append(record)
    
    status_tag = "[BEST]" if improved else f"[Patience {early_stopper.counter}/{PATIENCE}]"
    print(f"[Phase B - Epoch {epoch:02d}/{PHASE_B_EPOCHS}] Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc*100:.2f}% | Val Loss: {va_loss:.4f}, Val Acc: {va_acc*100:.2f}% | LR: {current_lr:.2e} {status_tag}")
    
    if early_stopper.should_stop:
        print(f"Early stopping triggered at epoch {epoch}.")
        break

# Restore best checkpoint
best_weights_path = OUTPUT_DIR / "_best_weights.pth"
model.load_state_dict(torch.load(best_weights_path, map_location=DEVICE))
model.eval()
print(f"Restored best model weights with Validation Loss: {early_stopper.best_loss:.4f}")

In [ ]:
# ── Cell 10: Temperature Scaling Confidence Calibration ───────────────────────
class TemperatureCalibrator(nn.Module):
    def __init__(self, base_model: nn.Module):
        super().__init__()
        self.model = base_model
        self.temperature = nn.Parameter(torch.tensor([1.5], device=DEVICE))

    def calibrate(self, loader: DataLoader) -> float:
        self.model.eval()
        nll_crit = nn.CrossEntropyLoss().to(DEVICE)
        logits_list, labels_list = [], []
        
        with torch.no_grad():
            for imgs, lbls in tqdm(loader, desc="Gathering Val Logits"):
                with autocast(device_type="cuda"):
                    logits_list.append(self.model(imgs.to(DEVICE)))
                labels_list.append(lbls.to(DEVICE))
                
        all_logits = torch.cat(logits_list).to(DEVICE)
        all_labels = torch.cat(labels_list).to(DEVICE)
        nll_before = nll_crit(all_logits, all_labels).item()

        lbfgs = optim.LBFGS([self.temperature], lr=0.01, max_iter=100)
        def eval_loss():
            lbfgs.zero_grad()
            loss = nll_crit(all_logits / self.temperature, all_labels)
            loss.backward()
            return loss
            
        lbfgs.step(eval_loss)
        with torch.no_grad():
            self.temperature.clamp_(0.05, 10.0)
            
        nll_after = nll_crit(all_logits / self.temperature, all_labels).item()
        learned_temp = self.temperature.item()
        print(f"Temperature Calibration Complete: T = {learned_temp:.4f}")
        print(f"  Val NLL: {nll_before:.4f} -> {nll_after:.4f}")
        return learned_temp

calibrator = TemperatureCalibrator(model)
LEARNED_TEMPERATURE = calibrator.calibrate(val_loader)

In [ ]:
# ── Cell 11: Held-out Test Set Evaluation & Metrics ───────────────────────────
model.eval()
all_preds, all_targets, all_probs = [], [], []

with torch.no_grad():
    for imgs, lbls in tqdm(test_loader, desc="Testing Held-out Split"):
        imgs = imgs.to(DEVICE)
        with autocast(device_type="cuda"):
            logits = model(imgs)
        scaled_logits = logits / LEARNED_TEMPERATURE
        probs = torch.softmax(scaled_logits, dim=1).cpu()
        all_probs.append(probs)
        all_preds.extend(probs.argmax(dim=1).numpy())
        all_targets.extend(lbls.numpy())

all_probs_cat = torch.cat(all_probs)
all_preds_arr = np.array(all_preds)
all_targets_arr = np.array(all_targets)

# Metrics
overall_acc = float((all_preds_arr == all_targets_arr).mean())
top3_indices = all_probs_cat.topk(3, dim=1).indices.numpy()
top3_acc = float((top3_indices == all_targets_arr[:, None]).any(axis=1).mean())

clf_report_dict = classification_report(all_targets_arr, all_preds_arr, target_names=class_dirs, output_dict=True, zero_division=0)
clf_report_str = classification_report(all_targets_arr, all_preds_arr, target_names=class_dirs, zero_division=0)

cm = confusion_matrix(all_targets_arr, all_preds_arr, labels=list(range(n_classes)))
confused_pairs = []
for r in range(n_classes):
    for c in range(n_classes):
        if r != c and cm[r, c] > 0:
            confused_pairs.append({"true_class": class_dirs[r], "predicted_class": class_dirs[c], "count": int(cm[r, c])})
confused_pairs.sort(key=lambda x: x["count"], reverse=True)
top10_confused = confused_pairs[:10]

low_recall_classes = [
    {"class": cls, "recall": round(clf_report_dict[cls]["recall"], 4), "precision": round(clf_report_dict[cls]["precision"], 4), "f1_score": round(clf_report_dict[cls]["f1-score"], 4), "support": int(clf_report_dict[cls]["support"])}
    for cls in class_dirs if cls in clf_report_dict and clf_report_dict[cls]["recall"] < 0.70
]

print("\n" + "=" * 75)
print(" CROP DISEASE CLASSIFIER — HELD-OUT TEST EVALUATION REPORT")
print("=" * 75)
print(f"Total Test Samples   : {len(all_targets_arr)}")
print(f"Overall Accuracy     : {overall_acc * 100:.2f}%")
print(f"Top-3 Accuracy       : {top3_acc * 100:.2f}%")
print(f"Calibrated Temp (T)  : {LEARNED_TEMPERATURE:.4f}")
print("\n--- Per-Class Metrics ---")
print(clf_report_str)

print("--- 10 Most Confused Class Pairs ---")
for k, p in enumerate(top10_confused, 1):
    print(f"  {k:2d}. True: '{p['true_class']}' -> Pred: '{p['predicted_class']}' ({p['count']} misclassifications)")

print("\n--- Classes with Recall < 70% ---")
if low_recall_classes:
    for itm in low_recall_classes:
        print(f"  - {itm['class']}: Recall={itm['recall']*100:.1f}%, Precision={itm['precision']*100:.1f}%, F1={itm['f1_score']*100:.1f}% (N={itm['support']})")
else:
    print("  None — all classes achieved >= 70% recall!")
print("=" * 75)

In [ ]:
# ── Cell 12: Save Confusion Matrix PNG ────────────────────────────────────────
plt.figure(figsize=(16, 14))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_dirs, yticklabels=class_dirs,
    linewidths=0.5, linecolor="#e0e0e0"
)
plt.title(f"Crop Disease Classifier Confusion Matrix (Acc: {overall_acc*100:.2f}%, Top-3: {top3_acc*100:.2f}%)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Class", fontsize=12)
plt.ylabel("True Class", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()

cm_png_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_png_path, dpi=300)
plt.close()
print(f"Confusion matrix heatmap saved to {cm_png_path}")

In [ ]:
# ── Cell 13: Save Checkpoint with 'trained': True Guard ───────────────────────
is_trained = overall_acc >= 0.85
if is_trained:
    print(f"SUCCESS: Held-out test accuracy ({overall_acc*100:.2f}%) exceeds target threshold of 85%.")
else:
    print(f"WARNING: Held-out test accuracy ({overall_acc*100:.2f}%) did not meet 85% target.")

checkpoint_payload = {
    "model_state_dict": model.state_dict(),
    "classes": class_dirs,
    "num_classes": n_classes,
    "temperature": float(LEARNED_TEMPERATURE),
    "architecture": "efficientnet_b0",
    "trained": is_trained,
    "test_accuracy": round(overall_acc, 6),
    "top3_accuracy": round(top3_acc, 6),
    "val_loss": round(early_stopper.best_loss, 6),
    "seed": SEED,
    "img_size": IMG_SIZE
}

checkpoint_file = OUTPUT_DIR / "disease_classifier_efficientnet_b0.pth"
torch.save(checkpoint_payload, checkpoint_file)
print(f"Checkpoint saved: {checkpoint_file} ({checkpoint_file.stat().st_size / (1024**2):.2f} MB)")
print(f"  'trained': {is_trained} | Accuracy: {overall_acc*100:.2f}%")

In [ ]:
# ── Cell 14: Save JSON Evaluation Report ──────────────────────────────────────
report_payload = {
    "dataset": "nirmalsankalana/crop-pest-and-disease-detection",
    "architecture": "efficientnet_b0",
    "trained": is_trained,
    "img_size": IMG_SIZE,
    "seed": SEED,
    "total_test_samples": int(len(all_targets_arr)),
    "overall_accuracy": round(overall_acc, 6),
    "top3_accuracy": round(top3_acc, 6),
    "temperature": round(float(LEARNED_TEMPERATURE), 6),
    "best_val_loss": round(early_stopper.best_loss, 6),
    "top10_most_confused_pairs": top10_confused,
    "low_recall_classes_under_70": low_recall_classes,
    "per_class_report": clf_report_dict,
    "training_history": {
        "phase_a": phase_a_history,
        "phase_b": phase_b_history
    }
}

report_file = OUTPUT_DIR / "evaluation_report.json"
with open(report_file, "w") as f:
    json.dump(report_payload, f, indent=2)
print(f"JSON report saved: {report_file}")

# Cleanup temporary best weights
temp_best = OUTPUT_DIR / "_best_weights.pth"
if temp_best.exists():
    temp_best.unlink()

print("\n--- Generated Artifacts in /kaggle/working ---")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name:<45} {p.stat().st_size / (1024**2):.2f} MB")